In [1]:
import pandas as pd
from transformers import pipeline
import torch

torch.cuda.is_available()

c:\Users\Kaylen\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1117 15:39:10.300000 32224 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


True

In [ ]:
# Read in my data (parquet file)
posts_data = pd.read_parquet("data/clean_posts.parquet") # text is in 'full_text' column
posts_data.rename(columns={'full_text': 'document_text', 'link_flair_text': 'post_flair'}, inplace=True)
posts_data['document_id'] = posts_data['post_id']
posts_data['document_type'] = "post"
posts_data['link'] = "https://www.reddit.com/comments/" + posts_data["post_id"]

comments_data = pd.read_parquet("data/clean_comments.parquet") # text is in 'body' column
comments_data.rename(columns={'body': 'document_text'}, inplace=True)
comments_data['document_id'] = comments_data['comment_id']
comments_data['link'] = "https://www.reddit.com" + comments_data['permalink']
comments_data['document_type'] = "comment"

# Assigning comment records a post_flair value based on the post they belong to.
comments_data = comments_data.merge(
    posts_data[["post_id", "post_flair"]],  # right table: id and post_flair (additional column after join)
    on="post_id", how="left", validate="many_to_one"
)

common_columns = list(set(posts_data.columns).intersection(set(comments_data.columns)))
keep_columns_and_order = [
    "document_id",
    "document_type",
    "post_id",
    "parent_id",
    "created_utc",
    "post_flair",
    "author",
    "mains",
    "document_text",
    "score",
    "num_comments",
    "link",
    "subreddit",
]

# Join posts + comments data
# Have to read on pandas join functions lol.

Common columns: ['document_id', 'score', 'link', 'created_utc', 'document_type', 'ups', 'post_flair', 'document_text', 'mains', 'author_flair_text', 'subreddit', 'author', 'post_id']
Keeping columns: ['document_id', 'document_type', 'post_id', 'parent_id', 'created_utc', 'post_flair', 'author', 'mains', 'document_text', 'score', 'num_comments', 'link', 'subreddit']


In [ ]:
emotion_pipeline = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions")

c:\Users\Kaylen\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kaylen\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Kaylen\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarni